# 05 — Evaluation and Explainability

**IBM Bob assisted** — SHAP integration code generated via IBM Bob Phase 4.

Produces: SHAP summary plot · SHAP waterfall plots (high / borderline / low probability)

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from dashboard.components.model_trainer import load_features, load_model, MODELS_DIR

PROC_DIR = Path('../data/processed')
plt.rcParams['figure.dpi'] = 120
shap.initjs()
print('Imports OK')

In [ ]:
X, y, feature_names = load_features()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f'X_test: {X_test.shape}')

In [ ]:
# Load best model (prefer xgboost, fallback to rf then lr)
best_name = None
for name in ['xgboost', 'random_forest', 'logistic_regression']:
    model_path = MODELS_DIR / f'{name}.joblib'
    if model_path.exists():
        best_model = load_model(name)
        best_name = name
        break

if best_name is None:
    raise RuntimeError('No trained models found — run notebook 04 first')

# Check metrics summary for the actual best
summary_path = MODELS_DIR / 'metrics_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    best_name = summary.get('best_model', best_name)
    best_model = load_model(best_name)

print(f'Best model: {best_name}')

In [ ]:
# Build SHAP explainer
import shap

# For pipelines (LR), extract the final estimator
def get_shap_explainer(model, X_train, feature_names):
    from sklearn.pipeline import Pipeline
    if isinstance(model, Pipeline):
        final = model[-1]
        X_transformed = model[:-1].transform(X_train)
        explainer = shap.LinearExplainer(final, X_transformed,
                                          feature_perturbation='interventional')
        return explainer, X_transformed, model[:-1].transform(X_test)
    else:
        explainer = shap.TreeExplainer(model)
        return explainer, X_train, X_test

explainer, X_train_t, X_test_t = get_shap_explainer(best_model, X_train, feature_names)
shap_values = explainer.shap_values(X_test_t)

# For binary classifiers shap_values may be list[2]; take index 1 (positive class)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')

In [ ]:
# SHAP Summary Plot (beeswarm)
plt.figure()
shap.summary_plot(sv, X_test_t, feature_names=feature_names, show=False)
plt.title(f'SHAP Summary — {best_name}', fontsize=13)
plt.tight_layout()
plt.savefig(PROC_DIR / 'shap_summary.png', bbox_inches='tight')
plt.show()
print('Saved shap_summary.png')

In [ ]:
# Find representative examples: high, borderline, low probability
if hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(X_test)[:, 1]
else:
    proba = np.zeros(len(X_test))

idx_high = int(np.argmax(proba))
idx_low  = int(np.argmin(proba))
idx_mid  = int(np.argmin(np.abs(proba - 0.5)))

print(f'High p={proba[idx_high]:.3f}  Mid p={proba[idx_mid]:.3f}  Low p={proba[idx_low]:.3f}')

In [ ]:
# SHAP waterfall plots
def waterfall(idx, label, sv_matrix, X_matrix, base_val):
    exp = shap.Explanation(
        values=sv_matrix[idx],
        base_values=base_val if np.isscalar(base_val) else base_val[0],
        data=X_matrix[idx],
        feature_names=feature_names
    )
    plt.figure(figsize=(9, 5))
    shap.plots.waterfall(exp, show=False, max_display=10)
    plt.title(f'SHAP Waterfall — {label}  (p={proba[idx]:.3f})', fontsize=12)
    plt.tight_layout()
    fname = PROC_DIR / f'shap_waterfall_{label.replace(" ","_")}.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

base = explainer.expected_value
if isinstance(base, list):
    base = base[1]

waterfall(idx_high, 'high_probability_go',  sv, X_test_t, base)
waterfall(idx_mid,  'borderline',            sv, X_test_t, base)
waterfall(idx_low,  'low_probability_scrub', sv, X_test_t, base)

In [ ]:
# ROC curve
from sklearn.metrics import RocCurveDisplay
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=ax, name=best_name)
ax.plot([0,1],[0,1],'k--',label='Random')
ax.set_title(f'ROC Curve — {best_name}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(PROC_DIR / 'roc_curve_best.png', bbox_inches='tight')
plt.show()